# 03 — Explainability

* SHAP TreeExplainer on the XGBoost component of the ensemble
* PyG `Explainer` (GNNExplainer) on the GNN node-classification head

Both used to satisfy EU AI Act 'transparency' / 'human oversight' clauses.

In [ ]:
import sys, pathlib, numpy as np, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))
from ml.data.synthetic_data import SyntheticConfig, generate
from ml.train_ensemble import (
    NUM_FEATURES, CAT_FEATURES, build_preprocessor, engineer_features,
)
import xgboost as xgb

df = engineer_features(generate(SyntheticConfig(n_transactions=4_000, seed=99)))
y = df['is_fraud'].values
pre = build_preprocessor()
X = pre.fit_transform(df[NUM_FEATURES + CAT_FEATURES])
model = xgb.XGBClassifier(max_depth=5, n_estimators=80, tree_method='hist').fit(X, y)

In [ ]:
import shap
expl = shap.TreeExplainer(model)
sv = expl.shap_values(X[:200])
shap.summary_plot(sv, features=X[:200], show=False)

## GNN explainability
Run the cell below if `torch_geometric` is installed. We use `Explainer` with `GNNExplainer` algorithm to surface the most influential edges for a flagged ring-member card.

In [ ]:
try:
    import torch
    from torch_geometric.explain import Explainer, GNNExplainer
    print('PyG available — instantiate Explainer(model, algorithm=GNNExplainer(epochs=100))')
except Exception as e:
    print('PyG not installed in this env:', e)